In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [19]:
class QuantizedLinear(nn.Module):
    """8-bit quantized linear layer"""
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.bias = nn.Parameter(torch.zeros(out_features))
        self.register_buffer('scale', torch.ones(1))
        
    def quantize_weight(self):
        """Quantize weights to 8-bit"""
        w = self.weight.data
        self.scale = w.abs().max() / 127.0
        quantized = torch.clamp(torch.round(w / self.scale), -128, 127)
        return quantized
    
    def forward(self, x):
        if self.training:
            return F.linear(x, self.weight, self.bias)
        else:
            q_weight = self.quantize_weight()
            dequantized = q_weight * self.scale
            return F.linear(x, dequantized, self.bias)


In [20]:
class KVCache:
    """Key-Value cache for efficient inference"""
    def __init__(self):
        self.k_cache = None
        self.v_cache = None
    
    def update(self, k, v):
        if self.k_cache is None:
            self.k_cache = k
            self.v_cache = v
        else:
            self.k_cache = torch.cat([self.k_cache, k], dim=2)
            self.v_cache = torch.cat([self.v_cache, v], dim=2)
        return self.k_cache, self.v_cache
    
    def clear(self):
        self.k_cache = None
        self.v_cache = None

In [21]:

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, use_quantization=True, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.dropout = nn.Dropout(dropout)
        
        LinearLayer = QuantizedLinear if use_quantization else nn.Linear
        
        self.q_proj = LinearLayer(d_model, d_model)
        self.k_proj = LinearLayer(d_model, d_model)
        self.v_proj = LinearLayer(d_model, d_model)
        self.out_proj = LinearLayer(d_model, d_model)
        
    def forward(self, x, mask=None, kv_cache=None):
        batch_size, seq_len, _ = x.shape
        
        q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Use KV cache if provided (for inference)
        if kv_cache is not None:
            k, v = kv_cache.update(k, v)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        attn_output = torch.matmul(attn_weights, v)
        
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.out_proj(attn_output)

In [22]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, use_quantization=True, dropout=0.1):
        super().__init__()
        LinearLayer = QuantizedLinear if use_quantization else nn.Linear
        
        self.fc1 = LinearLayer(d_model, d_ff)
        self.fc2 = LinearLayer(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        # Use GELU instead of ReLU for better gradients
        return self.fc2(self.dropout(F.gelu(self.fc1(x))))

In [23]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, use_quantization=True):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads, use_quantization, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, use_quantization, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None, kv_cache=None):
        # Pre-normalization for better training stability
        attn_out = self.attention(self.norm1(x), mask, kv_cache)
        x = x + self.dropout(attn_out)
        ff_out = self.ff(self.norm2(x))
        x = x + self.dropout(ff_out)
        return x

In [24]:
class QuantizedTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, num_heads=8, num_layers=6, 
                 d_ff=1024, max_seq_len=512, dropout=0.1, use_quantization=True):
        super().__init__()
        self.d_model = d_model
        self.use_quantization = use_quantization
        
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_seq_len, d_model)
        
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, dropout, use_quantization)
            for _ in range(num_layers)
        ])
        
        LinearLayer = QuantizedLinear if use_quantization else nn.Linear
        self.output = LinearLayer(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        batch_size, seq_len = x.shape
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, -1)
        
        x = self.token_embedding(x) + self.pos_embedding(positions)
        x = self.dropout(x)
        
        for layer in self.layers:
            x = layer(x, mask)
        
        return self.output(x)


In [25]:
class TextDataset(torch.utils.data.Dataset):
    """Dataset for text corpus"""
    def __init__(self, text, tokenizer, seq_len=128):
        self.tokenizer = tokenizer
        self.seq_len = seq_len
        self.tokens = tokenizer.encode(text)
        
    def __len__(self):
        return max(0, len(self.tokens) - self.seq_len)
    
    def __getitem__(self, idx):
        chunk = self.tokens[idx:idx + self.seq_len + 1]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y

In [26]:

class SimpleTokenizer:
    """Basic character or word-level tokenizer"""
    def __init__(self, corpus, level='char'):
        self.level = level
        if level == 'char':
            self.vocab = sorted(set(corpus))
        else:  # word level
            words = corpus.lower().split()
            self.vocab = sorted(set(words))
        
        self.vocab = ['<PAD>', '<UNK>'] + self.vocab
        self.token_to_id = {t: i for i, t in enumerate(self.vocab)}
        self.id_to_token = {i: t for i, t in enumerate(self.vocab)}
        
    def encode(self, text):
        if self.level == 'char':
            return [self.token_to_id.get(c, 1) for c in text]
        else:
            return [self.token_to_id.get(w, 1) for w in text.lower().split()]
    
    def decode(self, ids):
        if self.level == 'char':
            return ''.join([self.id_to_token.get(i, '<UNK>') for i in ids])
        else:
            return ' '.join([self.id_to_token.get(i, '<UNK>') for i in ids])
    
    @property
    def vocab_size(self):
        return len(self.vocab)

In [28]:
def train_model(model, dataloader, epochs=5, lr=0.001, warmup_steps=100):
    """Train the transformer model with gradient accumulation and mixed precision"""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler()  # Use torch.cuda.amp.GradScaler
    
    # Learning rate warmup and decay
    def lr_lambda(step):
        if step < warmup_steps:
            return step / warmup_steps
        return max(0.1, 0.5 * (1 + math.cos(math.pi * step / (epochs * len(dataloader)))))
    
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    model.train()
    accumulation_steps = 4  # Gradient accumulation
    
    for epoch in range(epochs):
        total_loss = 0
        optimizer.zero_grad()
        
        for batch_idx, (x, y) in enumerate(dataloader):
            x, y = x.to(device), y.to(device)
            
            # Mixed precision training
            # Use autocast only when CUDA is available; on CPU use a no-op context
            if device.type == 'cuda':
                autocast_ctx = torch.cuda.amp.autocast
            else:
                # torch.cuda.amp.autocast isn't applicable on CPU; use a no-op context manager
                from contextlib import nullcontext
                autocast_ctx = nullcontext
            with autocast_ctx():
                output = model(x)
                loss = criterion(output.view(-1, output.size(-1)), y.view(-1))
                loss = loss / accumulation_steps
            
            scaler.scale(loss).backward()
            
            if (batch_idx + 1) % accumulation_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
            
            total_loss += loss.item() * accumulation_steps
            
            if batch_idx % 10 == 0:
                print(f"Epoch {epoch+1}/{epochs}, Batch {batch_idx}, Loss: {loss.item()*accumulation_steps:.4f}, LR: {scheduler.get_last_lr()[0]:.6f}")
        
        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1} completed. Average Loss: {avg_loss:.4f}\n")
    
    return model.cpu()  # Move back to CPU for inference

def generate_text(model, tokenizer, start_text, max_len=100, temperature=1.0, top_k=50, top_p=0.9):
    """Generate text with top-k and top-p sampling"""
    model.eval()
    tokens = tokenizer.encode(start_text)
    max_seq = model.pos_embedding.num_embeddings  # Get max sequence length from model
    
    with torch.no_grad():
        for _ in range(max_len):
            # Only use last max_seq tokens to avoid index error
            input_tokens = tokens[-max_seq:] if len(tokens) > max_seq else tokens
            x = torch.tensor([input_tokens], dtype=torch.long)
            output = model(x)
            logits = output[0, -1, :] / temperature
            
            # Top-k filtering
            if top_k > 0:
                top_k_vals, top_k_indices = torch.topk(logits, min(top_k, logits.size(-1)))
                logits = torch.full_like(logits, float('-inf'))
                logits.scatter_(0, top_k_indices, top_k_vals)
            
            # Top-p (nucleus) sampling
            if top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[1:] = sorted_indices_to_remove[:-1].clone()
                sorted_indices_to_remove[0] = False
                indices_to_remove = sorted_indices[sorted_indices_to_remove]
                logits[indices_to_remove] = float('-inf')
            
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1).item()
            tokens.append(next_token)
    
    return tokenizer.decode(tokens)

# Example usage
if __name__ == "__main__":
    import sys
    
    # Load text from file
    if len(sys.argv) < 2:
        print("Usage: python script.py <text_file.txt>")
        print("\nUsing default sample text for demo...")
        corpus = "The quick brown fox jumps over the lazy dog. " * 20
    else:
        file_path = "alice.txt"
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                corpus = f.read()
            print(f"Loaded {len(corpus)} characters from {file_path}")
        except FileNotFoundError:
            print(f"Error: File '{file_path}' not found")
            sys.exit(1)
    
    # Create tokenizer
    tokenizer = SimpleTokenizer(corpus, level='char')
    print(f"Vocabulary size: {tokenizer.vocab_size}")
    
    # Create dataset and dataloader
    seq_len = 128
    dataset = TextDataset(corpus, tokenizer, seq_len)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=True)
    print(f"Training sequences: {len(dataset)}\n")
    
    # Create model
    model = QuantizedTransformer(
        vocab_size=tokenizer.vocab_size,
        d_model=128,
        num_heads=4,
        num_layers=4,
        max_seq_len=seq_len,
        use_quantization=True
    )
    
    print(f"Model size: {sum(p.numel() * p.element_size() for p in model.parameters()) / (1024*1024):.2f} MB\n")
    
    # Train model
    train_model(model, dataloader, epochs=5, lr=0.001)
    
    # Generate text
    start_text = corpus[:20] if len(corpus) >= 20 else corpus[:5]
    generated = generate_text(model, tokenizer, start_text, max_len=100)
    print(f"\nGenerated text:\n{generated}")
    
    # Save model
    torch.save({
        'model_state': model.state_dict(),
        'tokenizer_vocab': tokenizer.vocab,
        'tokenizer_level': tokenizer.level
    }, 'trained_model.pt')
    print("\nModel saved to 'trained_model.pt'")

Loaded 163916 characters from alice.txt
Vocabulary size: 93
Training sequences: 163788

Model size: 5.19 MB

Epoch 1/5, Batch 0, Loss: 15774.6367, LR: 0.000000
Epoch 1/5, Batch 10, Loss: 15976.0869, LR: 0.000020
Epoch 1/5, Batch 20, Loss: 15602.9492, LR: 0.000050
Epoch 1/5, Batch 30, Loss: 15648.8115, LR: 0.000070
Epoch 1/5, Batch 40, Loss: 15451.7773, LR: 0.000100
Epoch 1/5, Batch 50, Loss: 16022.6885, LR: 0.000120
Epoch 1/5, Batch 60, Loss: 15508.5439, LR: 0.000150
Epoch 1/5, Batch 70, Loss: 15465.7041, LR: 0.000170
Epoch 1/5, Batch 80, Loss: 15131.7002, LR: 0.000200
Epoch 1/5, Batch 90, Loss: 14708.7754, LR: 0.000220
Epoch 1/5, Batch 100, Loss: 14436.9053, LR: 0.000250
Epoch 1/5, Batch 110, Loss: 14344.8203, LR: 0.000270
Epoch 1/5, Batch 120, Loss: 13792.3955, LR: 0.000300
Epoch 1/5, Batch 130, Loss: 13952.2998, LR: 0.000320
Epoch 1/5, Batch 140, Loss: 13347.4141, LR: 0.000350
Epoch 1/5, Batch 150, Loss: 12962.1230, LR: 0.000370
Epoch 1/5, Batch 160, Loss: 12876.5654, LR: 0.000400
E

KeyboardInterrupt: 